# Handwriting OCR with a Vision-Language Model
 
 Russian handwriting recognition and evaluation with a quantized Qwen2.5-VL model.


**Задача 3: Распознавание рукописных текстов**

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_ID,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16
)

processor = AutoProcessor.from_pretrained(model_ID)

Дальше делал все по примеру из huggingface https://huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct#using-🤗--transformers-to-chat. Почти ничего не менял и все заработало, поместил анализ изображения в функцию.

In [67]:
from PIL import Image
from qwen_vl_utils import process_vision_info

#функция масштабирования, так как изображения очень большие, и без нее модель зависает
def auto_resize(image, ms=1280):
    w, h = image.size
    if max(w, h) > ms:
        r = ms / max(w, h)
        image = image.resize((int(w * r), int(h * r)))
    return image

def dec_image(image_path, prompt):
    img = Image.open(image_path).convert("RGB")
    img = auto_resize(img)
 
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": prompt}
            ]
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    image_inputs = process_vision_info(messages)[0]

    inputs = processor(
        text=[text],
        images=image_inputs,
        padding=True,
        return_tensors="pt"
    ).to(model.device)

    generated_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False
    )


    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    result = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True
    )[0].strip()

    return result

In [68]:
prompt = 'Извлеките рукописный текст из этого изображения. Верните только текст.'
print(dec_image('images/IMG_2694.jpg', prompt))

Современные языки программирования, такие как Python, Java и C++, позволяют решать широкий спектр задач - от создания веб-сайтов до разработки сложных систем искусственного интеллекта. Каждый язык обладает своей философией и областью применения, но все они основываются на общих принципах: алгоритмах, структурах данных и логике.


In [69]:
images = ['images/IMG_2694.jpg','images/IMG_2695.jpg', 'images/IMG_2696.jpg', 'images/IMG_2697.jpg', 'images/IMG_99998.jpg', 'images/IMG_99999.jpg']
hyps = []
for image in images:
    hyps.append(dec_image(image, prompt))
    print(hyps[-1])

Современные языки программирования, такие как Python, Java и C++, позволяют решать широкий спектр задач - от создания веб-сайтов до разработки сложных систем искусственного интеллекта. Каждый язык обладает своей философией и областью применения, но все они основываются на общих принципах: алгоритмах, структурах данных и логике.
Программирование - это не просто технический кавыч, а особый способ
мышления, позволяющий человеку создавать новые цифровые миры. В XXI веке
это стало универсальным языком взаимодействия с технологиями. Подобно тому,
как когда-то знание грамоты открывало доступ к культуре и науке, сегодня
умение писать код открывает путь к инновациям, автоматизации и творчеству.
История программирования начинается задолго до появления алгоритмов. Идея алгоритмов возникла в рамках математики, а в XIX веке первым в мире алгоритмом был создан первый компьютер. С появлением электронных вычислительных машин, основанных на принципах программирования, алгоритмы стали применяться в масс

In [70]:
# В hyps поместите выводы вашей модели, можете копирнуть руками, можете скриптом пройтись циклом по фото в датасете

from cer import calculate_cer_corpus




refs = ["""Современные языки программирования, такие как Python, Java и C++, позволяют решать широкий спектр задач — от создания веб-сайтов до разработки сложных систем искусственного интеллекта. Каждый язык обладает своей философией и областью применения, но все они основаны на общих принципах: алгоритмах, структурах данных и логике.""",
"""Программирование — это не просто технический навык, а особый способ мышления, позволяющий человеку создавать новые цифровые миры. В XXI веке оно стало универсальным языком взаимодействия с технологиями. Подобно тому как когда-то знание грамоты открывало доступ к культуре и науке, сегодня умение писать код открывает путь к инновациям, автоматизации и творчеству.""",
"""История программирования начинается задолго до появления современных компьютеров. Идеи алгоритмов можно проследить ещё в работах математиков прошлого, а в XIX веке Ада Лавлейс создала описание алгоритма для аналитической машины, став первым в мире программистом. С развитием вычислительной техники программирование превратилось в самостоятельную область знаний, объединяющую математику, инженерию и логику.""",
"""Это первая домашка на ИАДе.
Она, кажется, простая, но пока никто её не решил...""",
"""Сытость совсем не зависит от того,
сколько мы едим, а от того, как мы едим!
Так и счастье, так и счастье, Лёвушка,
оно вовсе не зависит от объёма
внешних благ, которые мы урвали у
жизни. Оно зависит только от нашего
отношения к ним! Об этом сказано ещё в
даосской этике: «Кто умеет
довольствоваться, тот всегда будет
доволен.""",
"При наличии уважительной причины дедлайн по д/з может быть перенесён. Дедлайн по д/з переносится на кол-во дней, равное продолжительности ув. причины."]

hyps = [sent.split() for sent in hyps]
refs = [sent.split() for sent in refs]

cer_corpus_score = calculate_cer_corpus(hyps, refs)
cer_corpus_score

{'count': 6,
 'mean': 0.1726564581403869,
 'median': 0.06310743367480956,
 'std': 0.2840128573898227,
 'min': 0.012658227848101266,
 'max': 0.7424242424242424,
 'cer_scores': [0.02127659574468085,
  0.016483516483516484,
  0.7424242424242424,
  0.012658227848101266,
  0.10493827160493827,
  0.13815789473684212]}

In [71]:
print(3 * min(1, 1 - cer_corpus_score["mean"] + 0.19))

3


Получилось не безупречно, но макс балл получен. Понадобилось только ресайзить картинку, с остальным модель справилась сама. Возможно я выбрал слишком хорошую модель, но ограничений никаких не было)

In [72]:
prompt = 'Что ты видишь на картинке?'
print(dec_image('images/IMG_2694.jpg', prompt))

На картинке изображена записка, написанная на белом листе бумаги синим чернилами. На бумаге написан текст на русском языке о современных языках программирования и их применении в различных областях. В центре записи есть большое красное "X", которое закрывает большую часть текста.


Прикольно